In [1]:
# =========================================================
# Notebook 03 — Feature Engineering and Panel Construction
# =========================================================
#
# Purpose:
# Build the final school-year modelling panel by combining
# cleaned source datasets, applying URN reconciliation,
# selecting the Progress 8 target, and generating lagged and
# delta features for downstream model training.
#
# Inputs:
# - master_data_cache.pkl
# - urn_mapping.pkl
#
# Stored in:
# - data/processed/
#
# Outputs:
# - school_panel_final.parquet
#
# Role in the pipeline:
# This notebook is the core feature-engineering stage of the
# pipeline. It transforms source-level administrative datasets
# into a longitudinal institutional panel suitable for
# regression and classification modelling.
#
# Reproducibility note:
# The file paths in this notebook are currently configured for
# the author’s local machine. These should be updated if the
# project is run in a different environment.
#
# Important:
# This notebook was developed on a local Windows environment.
# Users reproducing the pipeline should replace absolute paths
# with environment-specific paths or relative project paths.
#
# Methodological note:
# The final panel is built at the school-year level using:
# - reconciled institutional identifiers (URN_FINAL)
# - a selected Progress 8 target variable
# - structural features from census, absence, and workforce data
# - lagged features
# - delta (year-on-year change) features
#
# This notebook operationalises the dissertation’s focus on
# institutional momentum by constructing explicit change-based
# predictors in addition to static structural indicators.
# =========================================================

# =========================================================
# WP3 — FEATURE ENGINEERING (UPDATED: NO FRAGMENTATION)
# Drop-in replacement for the LAG/DELTA section to remove
# pandas "highly fragmented" PerformanceWarning.
# =========================================================

import pandas as pd
import numpy as np
import pickle
import os
import re

# =========================================================
# Configuration
# =========================================================
#
# This section defines the paths used to load the cached
# source data and URN mapping artefacts, and to save the
# final engineered panel.
#
# If reproducing this pipeline on another machine, update
# these paths before running the notebook.
# =========================================================

# --- CONFIGURATION ---
processed_folder = r'C:\Users\kiero\Documents\msc-dissertation\data\processed'
cache_path = os.path.join(processed_folder, 'master_data_cache.pkl')
map_path = os.path.join(processed_folder, 'urn_mapping.pkl')

print("🚀 Starting WP3: Feature Engineering (Robust + Type-Safe, No Fragmentation)...\n")

# =========================================================
# Step 1 — Load Cached Data and Mapping Artefacts
# =========================================================
#
# Inputs loaded here:
# - master_data_cache.pkl
# - urn_mapping.pkl
#
# The cache contains category-level datasets created in the
# ingestion stage, while the mapping file provides reconciled
# institutional identifiers for longitudinal continuity.
# =========================================================

# ---------------------------------------------------------
# 1) LOAD DATA
# ---------------------------------------------------------
with open(cache_path, 'rb') as f:
    dfs = pickle.load(f)
with open(map_path, 'rb') as f:
    urn_map = pickle.load(f)

# =========================================================
# Step 2 — Helper Functions
# =========================================================
#
# These functions standardise academic year values, clean and
# reconcile input datasets, and identify the most appropriate
# Progress 8 target variable from KS4 data.
#
# This flexibility is necessary because source schemas vary
# across years and datasets.
# =========================================================

# ---------------------------------------------------------
# 2) HELPERS
# ---------------------------------------------------------
def standardise_year_to_start_int(x):
    if pd.isna(x):
        return np.nan
    s = str(x).strip()

    if re.fullmatch(r"\d{4}", s):
        return int(s)
    if re.fullmatch(r"\d{6}", s):
        return int(s[:4])
    if re.fullmatch(r"\d{8}", s):
        return int(s[:4])

    m = re.fullmatch(r"(20\d{2})[/-](20\d{2})", s)
    if m:
        return int(m.group(1))

    m = re.fullmatch(r"(20\d{2})[/-](\d{2})", s)
    if m:
        return int(m.group(1))

    m = re.search(r"(20\d{2})", s)
    if m:
        return int(m.group(1))

    return np.nan

def clean_and_map(df, map_dict, urn_col='URN', year_col_candidates=('ACADEMICYEAR','YEAR')):
    if df is None:
        return None

    df = df.copy()
    df.columns = [c.upper().strip() for c in df.columns]

    # URN column
    if urn_col.upper() not in df.columns:
        for alias in ['SCHOOL_URN', 'URN', 'URN_FINAL']:
            if alias in df.columns:
                urn_col = alias
                break
    if urn_col.upper() not in df.columns:
        return None

    # Year column
    year_col = None
    for c in year_col_candidates:
        if c.upper() in df.columns:
            year_col = c.upper()
            break
    if year_col is None and 'ACADEMICYEAR' in df.columns:
        year_col = 'ACADEMICYEAR'
    if year_col is None:
        return None

    df['YEAR_START'] = df[year_col].apply(standardise_year_to_start_int)

    df['URN_FINAL'] = pd.to_numeric(df[urn_col.upper()], errors='coerce')
    df = df.dropna(subset=['URN_FINAL', 'YEAR_START'])
    df['URN_FINAL'] = df['URN_FINAL'].astype(int)

    # Apply mapping (int->int)
    df['URN_FINAL'] = df['URN_FINAL'].map(map_dict).fillna(df['URN_FINAL']).astype(int)
    df['YEAR_START'] = df['YEAR_START'].astype(int)

    return df

def pick_progress8_target(ks4_df):
    df = ks4_df.copy()
    cols = df.columns.tolist()

    preferred = ['P8MEA', 'P8', 'PROGRESS8', 'PROGRESS_8', 'PTGP8', 'P8_SCORE']
    for name in preferred:
        if name in cols:
            vals = pd.to_numeric(df[name], errors='coerce')
            if vals.notna().mean() > 0.2 and (-5 < vals.mean() < 5):
                df['TARGET_P8'] = vals
                return df, name

    potential = [c for c in cols if (('P8' in c) or ('PTGP8' in c)) and ('ATT8' not in c)]
    best_col = None
    best_score = -1

    for c in potential:
        vals = pd.to_numeric(df[c], errors='coerce')
        coverage = vals.notna().mean()
        mean_val = vals.mean()
        if coverage < 0.2:
            continue
        if not (-5 < mean_val < 5):
            continue
        if coverage > best_score:
            best_score = coverage
            best_col = c

    if best_col is None:
        raise ValueError("Could not identify a valid Progress 8 target column.")

    df['TARGET_P8'] = pd.to_numeric(df[best_col], errors='coerce')
    return df, best_col

# =========================================================
# Step 3 — Process Core Data Pillars
# =========================================================
#
# This step processes the major source datasets used in the
# modelling panel:
# - KS4 (target variable)
# - Census
# - Absence
# - School Workforce (SWF)
#
# Each dataset is:
# - cleaned
# - mapped to reconciled URNs
# - standardised to YEAR_START
# - aggregated to one row per school-year
# =========================================================

# ---------------------------------------------------------
# 3) PROCESS PILLARS
# ---------------------------------------------------------
print("2. Processing Data Pillars...")

print("   - Processing KS4 (Target)...")
ks4_raw = clean_and_map(dfs.get('ks4'), urn_map, urn_col='URN', year_col_candidates=('ACADEMICYEAR','YEAR','ACADEMIC_YEAR'))
ks4_raw, chosen_target = pick_progress8_target(ks4_raw)
print(f"     ✅ Selected Progress 8 column: {chosen_target}")

ks4 = ks4_raw[['URN_FINAL', 'YEAR_START', 'TARGET_P8']].dropna(subset=['TARGET_P8']).copy()
ks4 = ks4.groupby(['URN_FINAL', 'YEAR_START'], as_index=False).mean(numeric_only=True)
print(f"     [DEBUG] Retained KS4 Rows: {len(ks4):,}")

print("   - Processing Census...")
cen = clean_and_map(dfs.get('census'), urn_map, urn_col='URN', year_col_candidates=('ACADEMICYEAR','YEAR','ACADEMIC_YEAR'))
for c in cen.columns:
    if c not in ['URN_FINAL', 'YEAR_START']:
        cen[c] = pd.to_numeric(cen[c], errors='coerce')
cen = cen.groupby(['URN_FINAL', 'YEAR_START'], as_index=False).mean(numeric_only=True)
print(f"     [DEBUG] Census panel rows: {len(cen):,}")

print("   - Processing Absence...")
abs_data = clean_and_map(dfs.get('absence'), urn_map, urn_col='URN', year_col_candidates=('ACADEMICYEAR','YEAR','ACADEMIC_YEAR'))

abs_rate_col = 'PERCTOT' if 'PERCTOT' in abs_data.columns else next((c for c in abs_data.columns if ('PERC' in c and 'ABS' in c)), None)
if abs_rate_col:
    abs_data['ABSENCE_RATE'] = pd.to_numeric(abs_data[abs_rate_col], errors='coerce')
    abs_data = abs_data[['URN_FINAL', 'YEAR_START', 'ABSENCE_RATE']]
    abs_data = abs_data.groupby(['URN_FINAL', 'YEAR_START'], as_index=False).mean(numeric_only=True)
    print(f"     ✅ Using absence column: {abs_rate_col}")
else:
    abs_data = None
    print("     ⚠️ No absence rate column found; continuing without absence.")

print("   - Processing SWF (Workforce)...")
swf = clean_and_map(dfs.get('swf'), urn_map, urn_col='URN', year_col_candidates=('ACADEMICYEAR','YEAR','ACADEMIC_YEAR'))
workforce_cols = [c for c in ['TURNOVERRATE', 'TOTALTEACHERS', 'PTR'] if c in swf.columns]
for c in workforce_cols:
    swf[c] = pd.to_numeric(swf[c], errors='coerce')
swf = swf[['URN_FINAL', 'YEAR_START'] + workforce_cols]
swf = swf.groupby(['URN_FINAL', 'YEAR_START'], as_index=False).mean(numeric_only=True)
print(f"     [DEBUG] SWF panel rows: {len(swf):,}")

# =========================================================
# Step 4 — Merge School-Year Data into Master Panel
# =========================================================
#
# This step merges the processed source pillars into a single
# school-year panel keyed by:
# - URN_FINAL
# - YEAR_START
#
# KS4 is used as the core anchor table because it provides
# the target variable for modelling.
# =========================================================

# ---------------------------------------------------------
# 4) MERGE INTO MASTER PANEL
# ---------------------------------------------------------
print("\n3. Merging...")

master = ks4.merge(cen, on=['URN_FINAL', 'YEAR_START'], how='left')
if abs_data is not None:
    master = master.merge(abs_data, on=['URN_FINAL', 'YEAR_START'], how='left')
master = master.merge(swf, on=['URN_FINAL', 'YEAR_START'], how='left')

# =========================================================
# Step 5 — Generate Lagged and Delta Features
# =========================================================
#
# This step creates:
# - lagged predictors (feature_LAG1)
# - year-on-year change predictors (feature_DELTA)
# - TARGET_P8_LAG1 for the naive persistence baseline
#
# The delta features operationalise "institutional momentum"
# by capturing directional change rather than static context.
#
# The implementation attaches lag and delta matrices in one
# operation to avoid pandas fragmentation warnings.
# =========================================================

# ---------------------------------------------------------
# 5) LAGS + DELTAS (UPDATED: NO FRAGMENTATION)
# ---------------------------------------------------------
print("\n4. Calculating Lags/Delta (No-Fragmentation Mode)...")

master = master.sort_values(['URN_FINAL', 'YEAR_START']).reset_index(drop=True)

feature_cols = [c for c in master.columns if c not in ['URN_FINAL', 'YEAR_START', 'TARGET_P8']]
master[feature_cols] = master[feature_cols].apply(pd.to_numeric, errors='coerce')

g = master.groupby('URN_FINAL', sort=False)

# Lags
lags = g[feature_cols].shift(1)
lags.columns = [f"{c}_LAG1" for c in feature_cols]

# Deltas: (current - lag)
# Make lag column names align back to base names temporarily
lags_for_delta = lags.rename(columns={f"{c}_LAG1": c for c in feature_cols})
deltas = master[feature_cols] - lags_for_delta
deltas.columns = [f"{c}_DELTA" for c in feature_cols]

# Attach in one go (prevents fragmentation)
master = pd.concat([master, lags, deltas], axis=1)

# Optional naive baseline feature
master['TARGET_P8_LAG1'] = g['TARGET_P8'].shift(1)

# De-fragment explicitly
master = master.copy()

# =========================================================
# Step 6 — Save Final Modelling Panel
# =========================================================
#
# This step removes rows without valid target values,
# enforces identifier types, and saves the final panel as a
# parquet file for use in model training and evaluation.
# =========================================================

# ---------------------------------------------------------
# 6) SAVE
# ---------------------------------------------------------
print("\n5. Saving Final Panel...")

master = master.dropna(subset=['TARGET_P8']).copy()

master['URN_FINAL'] = pd.to_numeric(master['URN_FINAL'], errors='coerce')
before = len(master)
master = master.dropna(subset=['URN_FINAL'])
after = len(master)
print(f"   - Dropped {before - after} rows with invalid/missing URNs.")
master['URN_FINAL'] = master['URN_FINAL'].astype(int)

output_path = os.path.join(processed_folder, 'school_panel_final.parquet')
master.to_parquet(output_path, index=False)

print(f"\n✅ SUCCESS. Saved {len(master):,} rows to {output_path}")
print("   Next: run 04_model_training.ipynb")

🚀 Starting WP3: Feature Engineering (Robust + Type-Safe, No Fragmentation)...



UnpicklingError: pickle data was truncated

In [ ]:
# =========================================================
# Optional Verification Cell — Inspect Final Panel
# =========================================================
#
# Purpose:
# This cell performs a basic QA check on the final engineered
# panel by confirming:
# - dataset shape
# - year coverage
# - target distribution
# - naive baseline coverage
#
# Why this check is useful:
# It provides a quick confirmation that the panel was built
# successfully and that lagged target coverage exists for the
# persistence benchmark described in the dissertation.
#
# Note:
# This is a diagnostic / QA cell and is not required for the
# core pipeline to run.
# =========================================================

import pandas as pd

df = pd.read_parquet(r"C:\Users\kiero\Documents\msc-dissertation\data\processed\school_panel_final.parquet")

print("Shape:", df.shape)
print("Years:", sorted(df["YEAR_START"].unique())[:10], "...", sorted(df["YEAR_START"].unique())[-10:])
print(df["TARGET_P8"].describe())

# How many rows have a lagged target (for naive baseline)?
print("Naive baseline coverage:", df["TARGET_P8_LAG1"].notna().mean())

Shape: (21430, 262)
Years: [np.int64(2016), np.int64(2017), np.int64(2018), np.int64(2021), np.int64(2022), np.int64(2023)] ... [np.int64(2016), np.int64(2017), np.int64(2018), np.int64(2021), np.int64(2022), np.int64(2023)]
count    21430.000000
mean        -0.211383
std          0.760273
min         -4.120000
25%         -0.470000
50%         -0.080000
75%          0.260000
max          2.560000
Name: TARGET_P8, dtype: float64
Naive baseline coverage: 0.8189454036397573


In [ ]:
# =========================================================
# Optional Verification Cell — Inspect Feature List Content
# =========================================================
#
# Purpose:
# This cell inspects the saved risk-model feature list and
# prints feature names containing "SEN".
#
# Why this check is useful:
# It provides a quick sanity check that SEND-related features
# have been retained in the downstream modelling feature set,
# which is relevant to the dissertation’s discussion of
# SEND-related predictors.
#
# Note:
# This is a diagnostic / QA cell and is not required for the
# core feature-engineering pipeline to run.
# =========================================================

import joblib
PROCESSED_FOLDER = r"C:\Users\kiero\Documents\msc-dissertation\data\processed"
features = joblib.load(f"{PROCESSED_FOLDER}/risk_model_features.joblib")
[f for f in features if "SEN" in f.upper()][:30]


# =========================================================
# Outputs Produced
# =========================================================
#
# After successful execution, this notebook produces:
#
# - school_panel_final.parquet
#
# This file is the final longitudinal modelling panel used
# for:
# - model training
# - out-of-time evaluation
# - dashboard integration
#
# Next notebook in pipeline:
# - 04_model_training.ipynb
# =========================================================

['TEACHERS WITH AT LEAST ONE PERIOD OF SICKNESS ABSENCE (%)',
 'TOTAL NUMBER OF DAYS LOST TO SICKNESS ABSENCE',
 'AVERAGE (MEAN) NUMBER OF DAYS LOST TO TEACHER SICKNESS ABSENCE (ONLY TEACHERS IN SCHOOL TAKING SICKNESS ABSENCE)',
 'AVERAGE NUMBER OF DAYS LOST TO TEACHER SICKNESS ABSENCE (ALL TEACHERS IN SCHOOL)',
 'TSENELSE',
 'PSENELSE',
 'TSENELK',
 'PSENELK',
 'ABSENCE_RATE',
 'TEACHERS WITH AT LEAST ONE PERIOD OF SICKNESS ABSENCE (%)_LAG1',
 'TOTAL NUMBER OF DAYS LOST TO SICKNESS ABSENCE_LAG1',
 'AVERAGE (MEAN) NUMBER OF DAYS LOST TO TEACHER SICKNESS ABSENCE (ONLY TEACHERS IN SCHOOL TAKING SICKNESS ABSENCE)_LAG1',
 'AVERAGE NUMBER OF DAYS LOST TO TEACHER SICKNESS ABSENCE (ALL TEACHERS IN SCHOOL)_LAG1',
 'TSENELSE_LAG1',
 'PSENELSE_LAG1',
 'TSENELK_LAG1',
 'PSENELK_LAG1',
 'ABSENCE_RATE_LAG1',
 'TEACHERS WITH AT LEAST ONE PERIOD OF SICKNESS ABSENCE (%)_DELTA',
 'TOTAL NUMBER OF DAYS LOST TO SICKNESS ABSENCE_DELTA',
 'AVERAGE (MEAN) NUMBER OF DAYS LOST TO TEACHER SICKNESS ABSENCE (ONL